# SAR-Assisted Understanding — analysis walkthrough

Reads the artefacts written by `scripts/03_run_experiments.py` and `04_analyse.py`.
Run the pipeline first (`bash scripts/run_experiment.sh`), then execute this top to bottom.

**Question.** Does Sentinel-1 recover land-cover accuracy lost when Sentinel-2 is
partially obscured? The comparison that matters is **B vs C** at each masking level.


In [ ]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.config import load_config
from src.visualization import use_style
use_style()
M = ROOT / 'results' / 'metrics'
cfg = load_config(ROOT / 'configs' / 'config.yaml')
print('levels:', cfg.masking.levels, '| seeds:', cfg.train.seeds, '| threshold:', cfg.eval.threshold)


## 1. The data selection

BigEarthNet v2.0 is multi-label; we keep the 11 classes with >= 1000 positive patches.


In [ ]:
summary = json.load(open(M / 'subset_summary.json'))
manifest = pd.read_csv(M / 'subset_manifest.csv')
print(f"{summary['n_patches']} patches, {len(summary['classes'])} classes")
print('splits:', summary['split_sizes'])
print('audit problems:', summary['n_audit_problems'], 'of', summary['n_audited'], 'pairs checked')
pd.Series(summary['class_positives']).sort_values(ascending=False).to_frame('positives')


### Splits are spatially blocked, not random

Test is a contiguous interior region ringed by validation, ringed by train — so
neighbouring (highly correlated) patches almost never straddle train and test.


In [ ]:
sym = {'train': 0, 'validation': 1, 'test': 2}
fig, axes = plt.subplots(1, manifest.tile.nunique(), figsize=(11, 4.2))
for ax, (tile, g) in zip(np.atleast_1d(axes), manifest.groupby('tile')):
    grid = np.full((g.row.max()+1, g.col.max()+1), np.nan)
    grid[g.row, g.col] = g.split.map(sym)
    ax.imshow(grid, cmap='viridis', interpolation='nearest')
    ax.set_title(f'{tile}  (n={len(g)})'); ax.grid(False)

    idx = {(r, c): s for r, c, s in zip(g.row, g.col, g.split)}
    cross = tot = 0
    for (r, c), s in idx.items():
        for dr, dc in ((0, 1), (1, 0)):
            n = idx.get((r+dr, c+dc))
            if n is not None:
                tot += 1; cross += n != s
    print(f'{tile}: {cross}/{tot} adjacent pairs cross a split boundary ({cross/tot:.1%})')
plt.suptitle('Split assignment on the patch grid (dark=train, mid=val, light=test)')
plt.tight_layout(); plt.show()


## 2. The degradation is spatially coherent and exact

Masks are a deterministic function of `(patch_id, level, seed)`, so **arms B and C see
byte-identical degraded imagery** — the property the whole comparison rests on.


In [ ]:
from src.masking import generate_mask
levels = [float(v) for v in cfg.masking.levels]
fig, axes = plt.subplots(1, len(levels), figsize=(15, 2.7))
for ax, lv in zip(axes, levels):
    m = generate_mask('S2B_MSIL2A_20170808T094029_N9999_R036_T35ULA_20_20', lv,
                      sigma=cfg.masking.smoothing_sigma, mask_seed=cfg.masking.mask_seed,
                      mode=cfg.masking.mode)
    ax.imshow(m, cmap='gray_r'); ax.set_title(f'{lv:.0%}  (actual {m.mean():.1%})', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
plt.suptitle('Simulated cloud masks — coverage is exact by construction'); plt.tight_layout(); plt.show()


## 3. Headline result

Two regimes. **R1** trains on clean optical and evaluates degraded (the assignment's
minimum baseline). **R2** trains *both* arms with the same masking augmentation, so the
only difference between them is the SAR branch.


In [ ]:
gain = pd.read_csv(M / 'sar_gain_table.csv')
cols = ['masking_pct', 'B_degraded_optical', 'C_degraded_plus_sar', 'sar_gain',
        'sar_gain_boot_lo', 'sar_gain_boot_hi', 'control_fusion_shuffled_sar', 'D_sar_only']
for regime in ['clean', 'degraded']:
    print(f'--- {regime} ---')
    display(gain[gain.regime == regime][cols].set_index('masking_pct').round(4))


**Read this carefully.** In R1 the SAR gain climbs to +0.27 at 80% masking. In R2 it is
+0.037 at the same level. Most of the apparent benefit of SAR is actually the benefit of
*training on degraded data* — a confound that a B-vs-C comparison alone would hide.


In [ ]:
res = pd.read_csv(M / 'results.csv')
res.pivot_table(index=['regime', 'test_level'], columns='arm',
                values='macro_f1', aggfunc=['mean', 'std']).round(4)


### The capacity control

`fusion_shuf` is arm C with SAR features permuted across samples: identical parameter
count, no genuine pairing. It scores *below* optical-only everywhere, so arm C's
behaviour is not explained by having more parameters.


In [ ]:
d = res[res.regime == 'degraded']
for arm in ['optical', 'fusion', 'fusion_shuf', 'sar']:
    g = d[d.arm == arm].groupby('test_level').macro_f1.mean()
    plt.plot(g.index * 100, g.values, marker='o', label=arm)
plt.xlabel('% masked'); plt.ylabel('macro F1'); plt.legend(); plt.title('R2: all arms'); plt.show()


### Threshold-free view

Macro F1 at 100% collapses to ~0 because a constant input yields the class prior, which
sits below the 0.5 threshold. mAP shows what is actually retained.


In [ ]:
res[res.regime == 'degraded'].pivot_table(index='test_level', columns='arm',
                                          values='macro_map', aggfunc='mean').round(4)


## 4. SAR does not help all classes equally


In [ ]:
pcg = pd.read_csv(M / 'per_class_sar_gain.csv')
piv = pcg[pcg.regime == 'degraded'].pivot_table(index='class', columns='masking_pct', values='sar_gain')
piv.sort_values(80, ascending=False).round(3)


Inland waters gains at *every* level (calm water is a specular reflector — near-black and
unambiguous in SAR). Broad-leaved forest and Pastures are hurt at every operational level:
backscatter cannot separate canopy types that reflectance can.


## 5. Failure analysis

Cases are chosen by a rule fixed in advance, with both failure categories included by
construction. At 80% masking SAR repairs ~1.7% of patches and breaks ~3.0% — yet macro F1
still improves, because the repairs concentrate in rare classes that macro-averaging weights.


In [ ]:
from src.visualization import sample_f1
z = np.load(M / 'test_probs.npz', allow_pickle=True)
thr = float(cfg.eval.threshold)
t = z['targets']
pb, pc = z['degraded/optical/0.8'], z['degraded/fusion/0.8']
fb, fc = sample_f1(t, (pb >= thr).astype(int)), sample_f1(t, (pc >= thr).astype(int))
delta = fc - fb
print(f'SAR repairs : {(delta > 0.3).mean():.1%} of test patches')
print(f'SAR breaks  : {(delta < -0.3).mean():.1%}')
print(f'no change   : {(np.abs(delta) <= 0.3).mean():.1%}')
plt.hist(delta, bins=60, color='#eb6834'); plt.axvline(0, color='k', lw=1)
plt.xlabel('per-sample F1(C) - F1(B) at 80% masking'); plt.ylabel('patches'); plt.show()
pd.read_csv(M / 'qualitative_examples.csv')


## 6. Conclusion

1. Optical degradation is devastating for a model that never saw it (0.70 -> 0.28 at 80%)
   and mild for one that did (0.69 -> 0.60).
2. Against the fair baseline, SAR helps **only past ~60% cloud** (+0.037 at 80%), and is
   slightly *harmful* below that.
3. The benefit is concentrated in classes with distinctive backscatter — above all
   Inland waters (+0.17 to +0.23 at every level).
4. The hypothesis is therefore **partially supported**: SAR is complementary, but far less
   than the naive comparison suggests, and simple degradation-aware training buys about
   nine times more at 80% masking than SAR does.

See the README for limitations — chiefly that this subset spans only two Sentinel-2
acquisitions, which bounds how far any of these numbers generalise.


---

## 7. Additional experiment — SAR -> optical feature reconstruction (arm ED)

Arm C asks *"is SAR useful alongside degraded optical?"*. Arm ED asks the stronger
question: *"can SAR **predict** the optical representation a clean image would have
produced?"* The target is the 384-d ViT feature, never the Sentinel-2 pixels.

Run `scripts/08_encoder_decoder.py`, `09_ed_compare.py` and `10_ed_figures.py` first.
The arms A-D results above are untouched by this section.


In [ ]:
ed_rec  = pd.read_csv(M / 'ed_reconstruction.csv')
ed_mat  = pd.read_csv(M / 'ed_matched_comparison.csv')
ed_comp = pd.read_csv(M / 'ed_comparison.csv')
print('reconstruction rows:', len(ed_rec), ' variants:', sorted(ed_rec.variant.unique()))

### 7.1 The measurement trap

Raw cosine between `z_hat` and `z_clean` is 0.987, which looks like near-perfect
reconstruction. It is not. This feature space is strongly anisotropic, so a predictor
that learns *nothing* still scores high. Establish the floor before reading the number.

In [ ]:
zc_tr = np.load(ROOT / 'data/features/train_opt_L000.npy')
zc_te = np.load(ROOT / 'data/features/test_opt_L000.npy')
cos = lambda a, b: ((a*b).sum(1) / (np.linalg.norm(a,axis=1)*np.linalg.norm(b,axis=1)+1e-12))

mu = zc_tr.mean(0, keepdims=True)
perm = np.random.default_rng(0).permutation(len(zc_te))
print(f"cos(two unrelated patches)        = {cos(zc_te, zc_te[perm]).mean():.4f}   <- anisotropy floor")
print(f"cos(constant mean predictor)      = {cos(np.repeat(mu, len(zc_te), 0), zc_te).mean():.4f}")
print(f"cos(the decoder)                  = {ed_rec[ed_rec.variant=='ed'].cos_hat_vs_clean.mean():.4f}")
print(f"\nfraction of feature energy in the mean vector = "
      f"{np.linalg.norm(mu)**2 / (zc_te**2).sum(1).mean():.3f}")

So raw cosine is nearly useless here. The honest metrics are **mean-relative**:

- `R2 = 1 - MSE(z_hat, z_clean) / MSE(mean, z_clean)` — variance explained *beyond*
  the constant predictor. 0 = learned nothing.
- **centred cosine** — direction agreement after subtracting the training mean.

And the control that settles it: a decoder trained on **shuffled** SAR (rows permuted
against their targets), mirroring the `fusion_shuf` control used for arm C.

In [ ]:
g  = ed_rec[ed_rec.variant == 'ed'].groupby('test_level').mean(numeric_only=True)
gs = ed_rec[ed_rec.variant == 'ed_shuf'].groupby('test_level').mean(numeric_only=True)
pd.DataFrame({
    'R2': [g.r2_hat.mean(), gs.r2_hat.mean()],
    'centred cos': [g.centered_cos_hat.mean(), gs.centered_cos_hat.mean()],
    'raw cos': [g.cos_hat_vs_clean.mean(), gs.cos_hat_vs_clean.mean()],
}, index=['decoder on real SAR', 'decoder on shuffled SAR']).round(4)

The shuffled control collapses to exactly the constant-mean predictor (R2 ~ 0).
The real decoder explains **~46%** of patch-to-patch variance, so the mapping is
genuine and patch-specific — but the raw cosine overstated it badly.

### 7.2 Does the reconstruction beat the degraded feature?

`z_hat` is level-invariant (SAR is never masked), so its R2 is flat while the
degraded feature's decays — and goes sharply negative, meaning past ~40% masking the
ViT's own output is further from the clean feature than guessing the dataset mean.

In [ ]:
r = g[['r2_degraded', 'r2_hat']].copy()
r.index = (r.index * 100).astype(int)
r['z_hat closer?'] = np.where(r.r2_hat > r.r2_degraded, 'yes', 'no')
r.round(3)

### 7.3 Classification: B vs C vs ED

The first 3-seed run put ED above C at 80% by +0.007; an identical re-run gave -0.006.
MPS kernels are non-deterministic and the gap is smaller than the run-to-run spread, so
this was redone with **10 seeds, paired arm-to-arm** (`scripts/09_ed_compare.py`).

In [ ]:
ed_mat[['masking_pct','B_mean','C_mean','ED_mean','ED_minus_C','ED_minus_C_std',
        'ED_beats_C_n_seeds','seed_ci_excludes_zero']].round(4)

**ED beats fusion below 60% masking** (+0.017 to +0.024, 10/10 seeds), is
**indistinguishable at 80%** (+0.001, 5/10 seeds — do not claim a win here), and
**loses heavily at 100%** (-0.305).

Why it wins at low masking: arm C appends 2048 raw SAR dimensions to 384 optical ones and
the head must learn to ignore most of them — at 0% masking C is actually *worse* than
plain optical. ED compresses SAR into 384 dimensions already aligned with the optical
feature space, so the head sees a balanced input in one coordinate system.

Why it loses at 100%: `z_degraded` is constant, so half the ED input is dead, and the head
was trained only on levels <= 0.8. Shared design flaw with C, not a property of
reconstruction.

### 7.4 What the bottleneck costs

The classes where **raw fusion still beats reconstruction** are exactly those with
distinctive radar signatures that have no optical analogue.

In [ ]:
e = pd.read_csv(M / 'ed_per_class.csv')
e = e[(e.variant=='ed') & (e.regime=='ed_r2') & (e.test_level==0.8)][['class','support','f1']]
c = pd.read_csv(M / 'per_class.csv')
c = c[(c.regime=='degraded') & (c.arm=='fusion') & (c.test_level==0.8)][['class','f1']]
m = e.merge(c, on='class', suffixes=('_ED','_C'))
m['ED - C'] = (m.f1_ED - m.f1_C).round(3)
m.sort_values('ED - C', ascending=False).reset_index(drop=True)

Urban fabric (-0.057) is the clearest case: buildings produce a **double-bounce**
return that is radar-specific, so forcing SAR through a "predict the optical feature"
bottleneck necessarily discards it. Same for inland waters (specular return).

That is the conceptual cost of the reconstruction framing, and it shows up in the
numbers rather than needing to be argued.

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(ROOT / 'results/figures/06_encoder_decoder.png')))

### 7.5 Conclusion for the ED arm

1. **SAR genuinely predicts the clean optical representation** — R2 = 0.457 against a
   shuffled-SAR control at -0.005. But raw cosine (0.987) is a trap in this feature space.
2. Above ~15% masking the SAR-derived estimate is **closer to the clean feature than the
   masked image's own encoding is**.
3. Reconstruction **beats concatenation below 60% masking**, ties at 80%, and fails at 100%.
4. It systematically **loses the radar-only signal** (urban double-bounce, specular water)
   that direct fusion keeps.

Neither framing dominates. Reconstruction is the better use of SAR when optical is mostly
intact; concatenation is the better use when optical is nearly gone. A model that routed
between them on estimated cloud fraction would plausibly beat both — untested here.